# Worker Layer – the nearest workers for every cluster

The clustering model decides how many people each cluster needs (`teams.csv`: `C001 : 5 People`). This notebook
decides who: it fills every cluster with the nearest workers from the worker list.

**Reads** (never changes): `clusters.csv` in `RESULTS_FOLDER` (each cluster's team size and centre), and the worker
list `WORKER_FILE` with `staff_id, node, latitude, longitude` and optionally `partition`. `latitude, longitude` is the
worker's **base**, where the day starts; workers of one base share it. With `PARTITION` set, only that partition's
workers are used.

**Rules**

1. Every cluster gets exactly its team size.
2. A worker works in one cluster for the day.
3. The total distance from the workers' bases to their cluster centres, by road, is as small as possible.

With more workers than places, the ones needed least are `spare`. With fewer, the places that would need the longest
trips stay empty and their cluster shows them as `missing`.

**Writes** to `OUTPUT_FOLDER`:

| File | One row per | What is in it |
|---|---|---|
| `worker_assignments.csv` | worker | cluster, base, km to the cluster centre, `status` |
| `cluster_workers.csv` | cluster | team size, workers assigned, places empty, staff IDs, km |
| `worker_cabinets.csv` | worker and cabinet | `staff_id`, `CabinetId`, `type` (`Primary` or `Secondary`), `cluster_id` |
| `map.html` | | cluster centres, bases, and a line from each base to the clusters it staffs |

## Setup and parameters
Paths are relative to the project root. `RESULTS_FOLDER` is the clustering run to staff; `PARTITION` must be the
partition that run planned.

In [ ]:
RESULTS_FOLDER = "outputs"                 # the clustering results to staff (OUTPUT_FOLDER of ClusteringModel)
WORKER_FILE = "Dataset/mock_workers.csv"   # the worker list: staff_id, node, latitude, longitude, partition
PARTITION = "TRIS_1"                       # only this partition's workers (None = every worker in the list)
OUTPUT_FOLDER = "outputs/workers"          # results of the Worker Layer

DISTANCE = "road"        # "road": by road with OSRM, like the Distance Layer; "straight": straight line (no server)

# Road distances
OSRM_URL = "http://127.0.0.1:5000"  # our own OSRM server on this laptop: start it first with start_osrm.bat
OSRM_PROFILE = "driving"
OSRM_BATCH = 100        # max coordinates (bases + centres) per OSRM request
OSRM_RETRIES = 4        # attempts per request before giving up
ALLOW_FALLBACK = False  # True: if OSRM can't be reached, use straight-line x 1.4 instead (approximate!)
FALLBACK_FACTOR = 1.4   # only used when ALLOW_FALLBACK = True

In [ ]:
# Install any missing library into the Python that is running this notebook (e.g. Anaconda),
# so "Run All" works on a fresh machine. Already-installed libraries are left alone.
import importlib.util
import subprocess
import sys

REQUIRED = {"pandas": "pandas", "numpy": "numpy", "scipy": "scipy", "folium": "folium", "requests": "requests"}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing libraries:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("Python", sys.version.split()[0], "- all required libraries available")

In [ ]:
import time
from pathlib import Path

import folium
import numpy as np
import pandas as pd
import requests
from scipy.optimize import linear_sum_assignment

ROOT = Path.cwd()   # the project root
RESULTS = ROOT / RESULTS_FOLDER
OUT_DIR = ROOT / OUTPUT_FOLDER
OUT_DIR.mkdir(parents=True, exist_ok=True)
assert DISTANCE in ("road", "straight"), 'DISTANCE must be "road" or "straight"'

EARTH_RADIUS_KM = 6371.0088
START = time.time()
pd.set_option("display.width", 140)

## Load and check the data
`clusters.csv` needs `cluster_id`, `people`, `centre_lat` and `centre_long`. The worker list needs its four columns
(any upper or lower case), a unique `staff_id` on every row and valid coordinates; the notebook stops if not. These
checks run on the whole list, then `PARTITION` keeps that partition's workers. If the results say which partition
they are (`rules.csv`), it must be the same one.

In [ ]:
WORKER_COLUMNS = ["staff_id", "node", "latitude", "longitude"]


def read_workers(path, partition=None):
    # Read the worker list, give the four columns their usual names (any upper/lower case) and check them. With a
    # partition, only that partition's workers are kept (the checks run on the whole list first).
    raw = pd.read_csv(path, dtype=str)
    by_lower = {str(c).strip().lower(): c for c in raw.columns}
    missing = [c for c in WORKER_COLUMNS if c not in by_lower]
    if missing:
        raise ValueError(f"{path.name} has no column {', '.join(missing)}. Needed: {', '.join(WORKER_COLUMNS)}.")
    df = raw.rename(columns={by_lower[c]: c for c in WORKER_COLUMNS})[WORKER_COLUMNS].copy()
    df["staff_id"] = df["staff_id"].fillna("").str.strip()
    df["node"] = df["node"].fillna("").str.strip()
    for col in ["latitude", "longitude"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    problems = []
    if (df["staff_id"] == "").any():
        problems.append(f"{int((df['staff_id'] == '').sum())} row(s) have no staff_id")
    dup = df.loc[df["staff_id"].duplicated() & (df["staff_id"] != ""), "staff_id"].unique()
    if len(dup):
        problems.append(f"staff_id appears more than once: {', '.join(dup[:5])}")
    bad = df[["latitude", "longitude"]].isna().any(axis=1)
    if bad.any():
        problems.append(f"empty or non-numeric latitude/longitude for {', '.join(df.loc[bad, 'staff_id'].head(5))}")
    out = ~bad & (~df["latitude"].between(-90, 90) | ~df["longitude"].between(-180, 180))
    if out.any():
        problems.append(f"latitude/longitude out of range for {', '.join(df.loc[out, 'staff_id'].head(5))}")
    if problems:
        raise ValueError(f"{path.name}: " + "; ".join(problems))
    if partition is not None:
        if "partition" not in by_lower:
            raise ValueError(f"{path.name} has no partition column, so PARTITION must be None")
        keep = (raw[by_lower["partition"]].fillna("").str.strip() == partition).to_numpy()
        print(f"Partition {partition}: {int(keep.sum())} of {len(df)} workers in {path.name}")
        df = df[keep]
    return df.reset_index(drop=True)


def read_clusters(folder):
    # The clustering results: clusters.csv (each cluster's team size and centre).
    clusters = pd.read_csv(folder / "clusters.csv")
    missing = [c for c in ["cluster_id", "people", "centre_lat", "centre_long"] if c not in clusters.columns]
    if missing:
        raise ValueError(f"{folder / 'clusters.csv'} has no column {', '.join(missing)}: run the clustering "
                         "notebook (v2 or v4) first.")
    return clusters


clusters = read_clusters(RESULTS)
if PARTITION is not None and (RESULTS / "rules.csv").exists():
    planned = str(pd.read_csv(RESULTS / "rules.csv", dtype=str)["partition"].iloc[0])
    if planned != PARTITION:
        raise ValueError(f"{RESULTS_FOLDER} are the clusters of partition {planned}, not {PARTITION}")
workers = read_workers(ROOT / WORKER_FILE, PARTITION)
N_WORKERS, N_PLACES = len(workers), int(clusters["people"].sum())
N_BASES = len(workers.groupby(["latitude", "longitude"]))
print(f"{N_WORKERS} workers at {N_BASES} bases, from {workers['node'].nunique()} nodes ({WORKER_FILE})")
print(f"{len(clusters)} clusters need {N_PLACES} people per day ({RESULTS_FOLDER}/clusters.csv)")
if N_WORKERS > N_PLACES:
    print(f"{N_WORKERS - N_PLACES} more workers than places: they will be spare")
elif N_WORKERS < N_PLACES:
    print(f"WARNING: {N_PLACES - N_WORKERS} fewer workers than places: some places will stay empty")

## Distance from every base to every cluster centre
*Road* km come from OSRM, like in the Distance Layer: one request answers many bases and centres, and each base is
asked once however many workers it has. *Straight-line* km are always kept next to them. `DISTANCE` says which one
the matching uses. A pair with no road route is only chosen if nothing else is possible.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    # Straight-line km on the earth's surface; works on numbers and numpy arrays.
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(a))


OSRM_STATS = {"requests": 0, "fallback": False}


def fetch_table(srcs, dsts):
    # One OSRM Table request: road km from every (lat, lon) in srcs to every (lat, lon) in dsts.
    coords = ";".join(f"{lon:.6f},{lat:.6f}" for lat, lon in list(srcs) + list(dsts))   # OSRM wants lon,lat
    url = (f"{OSRM_URL}/table/v1/{OSRM_PROFILE}/{coords}"
           f"?sources={';'.join(str(i) for i in range(len(srcs)))}"
           f"&destinations={';'.join(str(i) for i in range(len(srcs), len(srcs) + len(dsts)))}"
           f"&annotations=distance")
    problem = ""
    for attempt in range(1, OSRM_RETRIES + 1):
        if attempt > 1:
            time.sleep(5 * 2 ** (attempt - 2))   # 5, 10, 20 s
        try:
            r = requests.get(url, timeout=60)
            answer = r.json() if "json" in r.headers.get("content-type", "") else {}
            if r.ok and answer.get("code") == "Ok":
                OSRM_STATS["requests"] += 1
                return np.array([[np.inf if m is None else m / 1000 for m in row] for row in answer["distances"]])
            problem = f"HTTP {r.status_code} {answer.get('code', '')} {answer.get('message', r.text[:200])}"
            if r.status_code == 400:
                break   # the request itself is wrong (e.g. too many points): retrying won't help
        except requests.Timeout:
            problem = "no answer within 60 s"
        except requests.ConnectionError:
            problem = "cannot connect to the server"
        except (requests.RequestException, ValueError) as err:
            problem = type(err).__name__
        print(f"  OSRM attempt {attempt}/{OSRM_RETRIES} failed: {problem}")
    raise ConnectionError(f"Could not get road distances from OSRM at {OSRM_URL} ({problem.strip()}). "
                          "If this is our own server, start it with start_osrm.bat; otherwise check OSRM_URL. "
                          "Or set ALLOW_FALLBACK = True to use approximate straight-line distances.")


def road_km_matrix(points, centres):
    # Road km from every base to every centre (points x centres), in requests of at most OSRM_BATCH
    # coordinates. inf = no road route. If OSRM fails and ALLOW_FALLBACK is on: straight-line x FALLBACK_FACTOR.
    km = np.full((len(points), len(centres)), np.inf)
    per_dst = max(1, min(len(centres), OSRM_BATCH // 2))
    try:
        for d0 in range(0, len(centres), per_dst):
            dsts = centres[d0:d0 + per_dst]
            per_src = OSRM_BATCH - len(dsts)
            for s0 in range(0, len(points), per_src):
                km[s0:s0 + per_src, d0:d0 + per_dst] = fetch_table(points[s0:s0 + per_src], dsts)
    except ConnectionError as err:
        if not ALLOW_FALLBACK:
            raise
        print("!" * 100 + f"\nWARNING: OSRM could not be reached, so road distances are APPROXIMATE "
              f"(straight-line x {FALLBACK_FACTOR}).\nReason: {err}\n" + "!" * 100)
        OSRM_STATS["fallback"] = True
        p, c = np.array(points), np.array(centres)
        km = FALLBACK_FACTOR * haversine_km(p[:, :1], p[:, 1:], c[None, :, 0], c[None, :, 1])
    return km


W_LAT, W_LON = workers["latitude"].to_numpy(), workers["longitude"].to_numpy()
C_LAT, C_LON = clusters["centre_lat"].to_numpy(), clusters["centre_long"].to_numpy()
straight = haversine_km(W_LAT[:, None], W_LON[:, None], C_LAT[None, :], C_LON[None, :])   # workers x clusters

if DISTANCE == "road":
    points = sorted(set(zip(W_LAT.round(6), W_LON.round(6))))     # each base once
    row_of = {p: i for i, p in enumerate(points)}
    by_point = road_km_matrix(points, list(zip(C_LAT, C_LON)))
    road = by_point[[row_of[p] for p in zip(W_LAT.round(6), W_LON.round(6))]]
    cost = road
else:
    road = np.full_like(straight, np.nan)                         # not measured
    cost = straight

nearest = np.nanmin(np.where(np.isfinite(cost), cost, np.nan), axis=1)
print(f"{N_WORKERS} workers x {len(clusters)} clusters, by {DISTANCE}"
      + (f": {OSRM_STATS['requests']} OSRM request(s)" if DISTANCE == "road" else "")
      + f"; nearest cluster centre is {np.nanmedian(nearest):.1f} km away for a typical worker "
        f"(farthest worker {np.nanmax(nearest):.1f} km)")
if DISTANCE == "road" and not np.isfinite(cost).all():
    print(f"Note: {int((~np.isfinite(cost)).sum())} worker-cluster pair(s) have no road route")

## Team places
A team of 5 becomes 5 **places**. All places of a cluster are the same distance from a given worker, so it doesn't
matter which one a worker takes.

In [ ]:
place_cluster = np.repeat(np.arange(len(clusters)), clusters["people"].to_numpy())   # cluster of every place
print(f"{len(place_cluster)} places in {len(clusters)} clusters "
      f"(teams of {clusters['people'].min()} to {clusters['people'].max()})")

## Match workers to places
We pair workers with places so the **total** km is as small as possible, one worker per place. This is the
assignment problem, solved exactly by the Hungarian method (`linear_sum_assignment` in `scipy`).

Giving out the shortest trip first, then the next, looks fair but can leave the last workers with very long trips.
The summary shows that "nearest first" total next to the Hungarian one.

In [ ]:
NO_ROUTE_KM = 1e6   # used in place of "no road route", so such a pair is chosen only if nothing else is possible


def match_workers(cost_km, place_cluster):
    # Hungarian method: the pairing of workers and places with the smallest total km.
    # Returns the cluster index for every worker (-1 = spare).
    km = np.where(np.isfinite(cost_km), cost_km, NO_ROUTE_KM)[:, place_cluster]   # workers x places
    rows, cols = linear_sum_assignment(km)
    cluster_of = np.full(len(cost_km), -1)
    cluster_of[rows] = place_cluster[cols]
    return cluster_of


def nearest_first(cost_km, place_cluster):
    # For comparison only: the shortest worker-cluster trip first, then the next shortest, and so on.
    km = np.where(np.isfinite(cost_km), cost_km, NO_ROUTE_KM)
    room = np.bincount(place_cluster, minlength=km.shape[1])
    cluster_of = np.full(len(km), -1)
    for flat in np.argsort(km, axis=None, kind="stable"):
        w, c = divmod(int(flat), km.shape[1])
        if cluster_of[w] == -1 and room[c] > 0:
            cluster_of[w], room[c] = c, room[c] - 1
    return cluster_of


def trip_km(cost_km, cluster_of):
    # km from each placed worker to their cluster centre (spare workers left out).
    placed = cluster_of >= 0
    return cost_km[np.where(placed)[0], cluster_of[placed]]


worker_cluster = match_workers(cost, place_cluster)
greedy_cluster = nearest_first(cost, place_cluster)
best, greedy = trip_km(cost, worker_cluster), trip_km(cost, greedy_cluster)
print(f"Hungarian method: {best.sum():,.1f} km in total, {best.mean():.1f} km on average, "
      f"longest trip {best.max():.1f} km ({DISTANCE})")
print(f"Nearest first:    {greedy.sum():,.1f} km in total, {greedy.mean():.1f} km on average, "
      f"longest trip {greedy.max():.1f} km")

## Worker and cluster tables
* `worker_assignments.csv`: one row per worker with their cluster, base, road and straight-line km and `status`.
* `cluster_workers.csv`: one row per cluster with its team size, workers `assigned`, empty places (`missing`), staff
  IDs, nodes, and the average and longest trip.
* `worker_cabinets.csv`: one row per worker and cabinet. `Primary` = every cabinet of the worker's cluster,
  `Secondary` = the cluster's nearby backup cabinets (`secondaries.csv`). Spare workers have no rows.

In [ ]:
def worker_table(workers, clusters, worker_cluster, road, straight):
    # One row per worker, assigned workers by cluster first, spare workers at the end.
    placed = worker_cluster >= 0
    idx = np.where(placed, worker_cluster, 0)
    rows = np.arange(len(workers))
    table = workers[["staff_id", "node"]].copy()
    table["cluster_id"] = np.where(placed, clusters["cluster_id"].to_numpy()[idx], "")
    table["latitude"], table["longitude"] = workers["latitude"], workers["longitude"]
    table["road_km_to_centre"] = np.where(placed, road[rows, idx], np.nan)
    table["straight_km_to_centre"] = np.where(placed, straight[rows, idx], np.nan)
    table["status"] = np.where(placed, "assigned", "spare")
    return (table.sort_values(["status", "cluster_id", "staff_id"], key=lambda s: s.map({"assigned": 0, "spare": 1})
                              if s.name == "status" else s, kind="stable").reset_index(drop=True))


def cluster_table(clusters, table):
    # One row per cluster: who works there and how far they travel.
    placed = table[table["status"] == "assigned"]
    group = placed.groupby("cluster_id")
    out = clusters[["cluster_id", "people"]].copy()
    out["assigned"] = out["cluster_id"].map(group.size()).fillna(0).astype(int)
    out["missing"] = out["people"] - out["assigned"]
    out["staff_ids"] = out["cluster_id"].map(group["staff_id"].agg("; ".join)).fillna("")
    out["nodes"] = out["cluster_id"].map(group["node"].agg(lambda s: ", ".join(sorted(set(s))))).fillna("")
    for kind in ["road", "straight"]:
        out[f"mean_{kind}_km"] = out["cluster_id"].map(group[f"{kind}_km_to_centre"].mean())
        out[f"max_{kind}_km"] = out["cluster_id"].map(group[f"{kind}_km_to_centre"].max())
    out["centre_lat"], out["centre_long"] = clusters["centre_lat"], clusters["centre_long"]
    return out


def cabinet_table(table, folder):
    # One row per assigned worker and cabinet: every cabinet of the worker's cluster (Primary), then the cluster's
    # backup cabinets (Secondary), from the clustering results in `folder`.
    work = [pd.read_csv(folder / "assignments.csv", usecols=["CabinetId", "cluster_id"], dtype=str)
            .assign(type="Primary")]
    if (folder / "secondaries.csv").exists():
        work.append(pd.read_csv(folder / "secondaries.csv", usecols=["cluster_id", "CabinetId"], dtype=str)
                    .assign(type="Secondary"))
    placed = table.loc[table["status"] == "assigned", ["staff_id", "cluster_id"]]
    out = placed.merge(pd.concat(work, ignore_index=True), on="cluster_id")[["staff_id", "CabinetId", "type",
                                                                             "cluster_id"]]
    return out.sort_values(["cluster_id", "staff_id", "type", "CabinetId"], kind="stable").reset_index(drop=True)


worker_assignments = worker_table(workers, clusters, worker_cluster, road, straight)
cluster_workers = cluster_table(clusters, worker_assignments)
worker_cabinets = cabinet_table(worker_assignments, RESULTS)
worker_assignments.head(8).round(2)

## Validation
The notebook stops if any of these fails: every worker appears once and is in at most one cluster; no cluster gets
more than its team; the number of places filled is the smaller of workers and places; the Hungarian total is not
above the nearest-first total; and every placed worker has exactly their cluster's cabinets as `Primary`, none of them
twice.

In [ ]:
def validate(workers, clusters, worker_assignments, cluster_workers, worker_cabinets, best, greedy):
    assert worker_assignments["staff_id"].is_unique, "a worker appears twice"
    assert set(worker_assignments["staff_id"]) == set(workers["staff_id"]), "a worker is missing from the table"
    assigned = worker_assignments[worker_assignments["status"] == "assigned"]
    assert assigned["cluster_id"].isin(clusters["cluster_id"]).all(), "a worker is in an unknown cluster"
    assert (cluster_workers["assigned"] <= cluster_workers["people"]).all(), "a cluster has too many workers"
    assert (cluster_workers["assigned"] + cluster_workers["missing"] == cluster_workers["people"]).all()
    assert len(assigned) == min(len(workers), int(clusters["people"].sum())), "wrong number of places filled"
    assert best.sum() <= greedy.sum() + 1e-6, "the matching is worse than nearest first"
    primary = worker_cabinets[worker_cabinets["type"] == "Primary"]
    size = pd.read_csv(RESULTS / "assignments.csv", dtype=str).groupby("cluster_id").size()
    got = primary.groupby("staff_id").size().reindex(assigned["staff_id"]).fillna(0)
    assert (got.to_numpy() == assigned["cluster_id"].map(size).to_numpy()).all(), \
        "a worker does not have exactly the cabinets of their cluster"
    assert not worker_cabinets.duplicated(["staff_id", "CabinetId"]).any(), "a cabinet is listed twice for a worker"
    print(f"All checks passed: {len(assigned)} workers placed, {int((worker_assignments['status'] == 'spare').sum())} "
          f"spare, {int(cluster_workers['missing'].sum())} place(s) empty")


validate(workers, clusters, worker_assignments, cluster_workers, worker_cabinets, best, greedy)

### Result summary

In [ ]:
placed = worker_assignments[worker_assignments["status"] == "assigned"]
km_col = f"{DISTANCE}_km_to_centre"
short = cluster_workers[cluster_workers["missing"] > 0]
staffed = cluster_workers[cluster_workers["assigned"] > 0]
summary = pd.Series({
    "Clusters": len(clusters),
    "Places (people needed per day)": N_PLACES,
    "Workers in the list": N_WORKERS,
    "Workers placed": len(placed),
    "Spare workers": int((worker_assignments["status"] == "spare").sum()),
    "Places left empty": int(short["missing"].sum()),
    f"Total km to cluster centres ({DISTANCE})": f"{best.sum():,.1f} (nearest first: {greedy.sum():,.1f})",
    "Average trip (km)": f"{placed[km_col].mean():.1f}",
    "Longest trip (km)": f"{placed[km_col].max():.1f}",
    "Workers within 10 km of their cluster": f"{(placed[km_col] <= 10).mean():.0%}",
    "Clusters staffed from one node only": f"{(staffed['nodes'].str.count(',') == 0).mean():.0%}",
    "Worker-cabinet rows (primary / secondary)": f"{(worker_cabinets['type'] == 'Primary').sum()} / "
                                                 f"{(worker_cabinets['type'] == 'Secondary').sum()}",
    "Run time": f"{time.time() - START:.0f} s",
})
print(summary.to_string())
if OSRM_STATS["fallback"]:
    print("WARNING: road km are APPROXIMATE (OSRM could not be reached)")
if len(short):
    print("Clusters with empty places: " + ", ".join(f"{c} ({m} missing)" for c, m in
                                                    zip(short["cluster_id"], short["missing"])))
if (placed[km_col] >= NO_ROUTE_KM).any():
    print("WARNING: some workers have no road route to their cluster: " +
          ", ".join(placed.loc[placed[km_col] >= NO_ROUTE_KM, "staff_id"]))
cluster_workers[["cluster_id", "people", "assigned", "missing", "staff_ids", "nodes", f"mean_{DISTANCE}_km",
                 f"max_{DISTANCE}_km"]].round(1)

## Save outputs
The three tables go to `OUTPUT_FOLDER`. The staff IDs of a cluster are separated by `; `, and distances are rounded
to 1 m.

In [ ]:
def round_km(table):
    # km to 3 decimals (1 m); latitude and longitude are kept exactly as they came in.
    return table.round({c: 3 for c in table.columns if c.endswith("_km") or c.endswith("_km_to_centre")})


round_km(worker_assignments).to_csv(OUT_DIR / "worker_assignments.csv", index=False)
round_km(cluster_workers).to_csv(OUT_DIR / "cluster_workers.csv", index=False)
worker_cabinets.to_csv(OUT_DIR / "worker_cabinets.csv", index=False)
print(f"saved worker_assignments.csv ({len(worker_assignments)} rows), cluster_workers.csv "
      f"({len(cluster_workers)} rows) and worker_cabinets.csv ({len(worker_cabinets)} rows) in {OUTPUT_FOLDER}")

## Interactive map
`map.html` in `OUTPUT_FOLDER`, also shown below.

* **Cluster centres** with the team size in them; a red ring means empty places.
* **Bases**: one square per base with its number of workers, in the colour of the cluster they go to (dark grey: more
  than one cluster, light grey: all spare).
* **Base to cluster**: a straight line from each base to every cluster it staffs, thicker for more workers. The km in
  the popups are by road.

In [ ]:
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SPARE_FILL = "#9aa0a6"
MIXED_FILL = "#3d4160"   # a base whose workers go to more than one cluster


def people_text(n):
    return f"{n} {'person' if n == 1 else 'people'}"


def centre_icon(team, colour, short):
    # A large round marker in the cluster's colour with the team size in it (red ring = empty places).
    ring = "#d00000" if short else "#fff"
    return folium.DivIcon(icon_size=(32, 32), icon_anchor=(16, 16),
                          html=f"<div style='width:24px;height:24px;border-radius:50%;background:{colour};"
                               f"border:4px solid {ring};box-shadow:0 0 0 2px #111,0 2px 8px rgba(0,0,0,.6);"
                               f"color:#fff;font:bold 15px/24px sans-serif;text-align:center;"
                               f"text-shadow:0 0 3px #000'>{team}</div>")


def name_icon(cid, team):
    return folium.DivIcon(icon_size=(150, 20), icon_anchor=(-20, 10),
                          html=f"<div style='display:inline-block;white-space:nowrap;background:#fff;"
                               f"border:1px solid #111;border-radius:3px;padding:0 5px;"
                               f"font:bold 12px/18px sans-serif;color:#111'>{cid} : {people_text(team)}</div>")


def base_icon(n, colour):
    # A square marker with the number of workers at this base.
    return folium.DivIcon(icon_size=(26, 26), icon_anchor=(13, 13),
                          html=f"<div style='width:20px;height:20px;border-radius:4px;background:{colour};"
                               f"border:3px solid #111;color:#fff;font:bold 12px/20px sans-serif;text-align:center;"
                               f"text-shadow:0 0 3px #000'>{n}</div>")


def make_map(cluster_workers, worker_assignments):
    fmap = folium.Map(location=[cluster_workers["centre_lat"].mean(), cluster_workers["centre_long"].mean()],
                      zoom_start=9, tiles="OpenStreetMap")
    colours = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(cluster_workers["cluster_id"])}
    centre_layer = folium.FeatureGroup(name="Cluster centres")
    name_layer = folium.FeatureGroup(name="Cluster names")
    base_layer = folium.FeatureGroup(name="Bases")
    line_layer = folium.FeatureGroup(name="Base to cluster")
    info = cluster_workers.set_index("cluster_id")

    for cid, p in info.iterrows():
        staff = p["staff_ids"].replace("; ", ", ") or "none"
        popup = (f"<b>{cid}</b>: team of {p['people']}<br>"
                 f"centre lat, long: {p['centre_lat']:.5f}, {p['centre_long']:.5f}<br>"
                 f"workers: {staff}<br>nodes: {p['nodes'] or '-'}")
        if p["assigned"]:
            popup += (f"<br>average trip: {p[f'mean_{DISTANCE}_km']:.1f} km, "
                      f"longest {p[f'max_{DISTANCE}_km']:.1f} km ({DISTANCE})")
        if p["missing"]:
            popup += f"<br><b>{p['missing']} place(s) empty</b>: not enough workers"
        folium.Marker([p["centre_lat"], p["centre_long"]], icon=centre_icon(p["people"], colours[cid], p["missing"]),
                      tooltip=f"{cid} : {p['assigned']} of {people_text(p['people'])}", z_index_offset=1000,
                      popup=folium.Popup(popup, max_width=300)).add_to(centre_layer)
        folium.Marker([p["centre_lat"], p["centre_long"]], icon=name_icon(cid, p["people"]),
                      interactive=False).add_to(name_layer)

    for (lat, lon), here in worker_assignments.groupby(["latitude", "longitude"], sort=False):
        placed = here[here["status"] == "assigned"]
        lines = [f"<b>Base</b> of node {', '.join(sorted(here['node'].unique()))}: {len(here)} workers<br>"
                 f"lat, long: {lat:.5f}, {lon:.5f}"]
        for cid, team in placed.groupby("cluster_id"):
            p = info.loc[cid]
            km = team[f"{DISTANCE}_km_to_centre"].iloc[0]
            lines.append(f"{cid} ({km:.1f} km, {DISTANCE}): {', '.join(team['staff_id'])}")
            folium.PolyLine([[lat, lon], [p["centre_lat"], p["centre_long"]]], color=colours[cid],
                            weight=2 + 1.5 * len(team), opacity=0.75,
                            tooltip=f"{len(team)} worker(s) → {cid}").add_to(line_layer)
        spare = here.loc[here["status"] == "spare", "staff_id"]
        if len(spare):
            lines.append(f"<b>spare</b>: {', '.join(spare)}")
        goes_to = list(placed["cluster_id"].unique())
        colour = SPARE_FILL if not goes_to else colours[goes_to[0]] if len(goes_to) == 1 else MIXED_FILL
        folium.Marker([lat, lon], icon=base_icon(len(here), colour), z_index_offset=500,
                      tooltip=f"base: {len(here)} workers → " + (", ".join(goes_to) or "spare"),
                      popup=folium.Popup("<br>".join(lines), max_width=320)).add_to(base_layer)

    for layer in (line_layer, base_layer, name_layer, centre_layer):
        layer.add_to(fmap)
    folium.LayerControl(collapsed=False).add_to(fmap)
    lats = pd.concat([cluster_workers["centre_lat"], worker_assignments["latitude"]])
    lons = pd.concat([cluster_workers["centre_long"], worker_assignments["longitude"]])
    fmap.fit_bounds([[lats.min(), lons.min()], [lats.max(), lons.max()]])   # open zoomed to the bases and centres
    n_spare = int((worker_assignments["status"] == "spare").sum())
    title = (f"<div style='position:fixed;bottom:24px;left:10px;z-index:9999;background:white;padding:6px 10px;"
             f"border-radius:4px;font:14px sans-serif;box-shadow:0 1px 4px rgba(0,0,0,.3)'>"
             f"<b>{len(cluster_workers)} clusters, {int(cluster_workers['assigned'].sum())} of "
             f"{int(cluster_workers['people'].sum())} places filled</b><br>"
             f"{len(worker_assignments)} workers at {N_BASES} bases, {n_spare} spare, "
             f"{int(cluster_workers['missing'].sum())} place(s) empty; nearest by {DISTANCE}</div>")
    fmap.get_root().html.add_child(folium.Element(title))
    return fmap


fmap = make_map(cluster_workers, worker_assignments)
fmap.save(OUT_DIR / "map.html")
print(f"saved {OUTPUT_FOLDER}/map.html")
fmap

## Notes and limitations
* A trip is measured from the base to the cluster centre, not along the day's route.
* The matching is for one day: it doesn't spread long trips over a week or know about shifts and leave.
* Every worker can work in every cluster of their partition. `node` is shown but is not a rule; a node rule, a travel
  limit or skills can be added later by changing the distances the matching uses.
* `Dataset/mock_workers.csv` is made up by `tools/generate_mock_workers.ipynb`; use the real list for real runs.